# 01 — The car**Before you can control something, you have to know what it does on its own.**There is no controller in this notebook. We build the vehicle from a force balance, solve thatbalance on paper for the numbers the rest of the lecture leans on, and then run the car openloop — constant throttle, no feedback anywhere — to see what it actually does.The notebook ends by reading three numbers off a single step response: the gain `K`, the timeconstant `tau`, and the dead time `theta`. Notebook 08 turns those three numbers intocontroller gains without looking at the model again, which is the whole point of measuringthem.Speeds are in km/h everywhere. The model emits km/h through `Vehicle.ToKmPerHour`, a blockthat is visible in the diagram, so nothing in this notebook converts a plotted signal afterthe fact.

## Setup`support.jl` carries the boilerplate shared by all ten notebooks: the environment bootstrap,the parameter set, the sweep machinery, and the plotting and readout conventions. It carriesno physics. Every derivation in this lecture stays in the notebook, where it can be read.It comes in two steps. The first is plain Julia and needs nothing installed. The second bringsup the Dyad environment and loads the component library, which takes a minute on a cold start.

In [ ]:
include("support.jl")using .Lecture01Support

In [ ]:
setup()

## 1. What a system is**Inputs, outputs, state, disturbances.**| | For this car ||---|---|| **Input** — what we are allowed to change | the torque commanded of the engine || **Output** — what we can measure | road speed, in km/h || **State** — what the system remembers | how fast it is already going, and how full the engine's manifold is || **Disturbance** — what moves the output without asking | the road gradient |Control is the business of choosing the input so that the output does what we want while thedisturbances do as they like. Everything that follows in this lecture is a variation on thatone sentence.The split is a modeling choice rather than a property of the car. The gradient is adisturbance because we cannot command it, not because it is small — on the 10% climb innotebook 06 it is the largest force in the balance below, larger than drag and rollingresistance together.

## 2. The force balanceNewton's second law along the direction of travel:$$m\,\frac{dv}{dt}\;=\;\underbrace{\frac{T\,i}{r}}_{\text{tractive}}\;-\;\underbrace{\tfrac{1}{2}\,\rho\,C_{d}A\,v^{2}}_{O_V}\;-\;\underbrace{f_{r}\,m\,g\,\cos\alpha}_{O_f}\;-\;\underbrace{m\,g\,\sin\alpha}_{O_s}$$In slide 16's notation: $O_V$ is drag resistance, $O_f$ is friction (rolling) resistance and$O_s$ is the resistance to overcome a slope. Two of the slide's five terms are not writtenhere. $O_z$, the acceleration resistance, is the inertia of the rotating masses — at thislevel of fidelity it is absorbed into $m$, and it reappears as a term of its own in notebook10 when the wheel is given its own inertia. $O_{aux}$, the auxiliary electrical loads, neverappears in this lecture at all.The tractive term is the driveline: engine torque $T$ multiplied by the gear ratio $i$ anddivided by the wheel radius $r$. The Dyad model keeps the gear and the wheel as separatecomponents so that the block diagram reads the same way this line does.Three things are worth noticing before we put numbers in.- **Drag is the only term that grows with speed**, and it grows as the square. That is what  makes a terminal speed exist at all.- **Rolling resistance is very nearly constant.** It depends on the gradient only through a  cosine, which is 0.995 even on a 10% slope.- **Nothing here is linear.** The $v^2$ makes sure of it. The car is close enough to linear  over any 20 km/h window that the linear theory in the rest of the lecture applies, and that  is an approximation we are choosing, not a fact about cars.

### Terminal speedSet $\frac{dv}{dt} = 0$ on flat road at full torque. The gradient term vanishes, the cosine isone, and what is left is a quadratic in $v$:$$\frac{T_{max}\,i}{r} \;=\; \tfrac{1}{2}\rho C_{d}A\,v^{2} \;+\; f_{r}\,m\,g\qquad\Longrightarrow\qquadv_{term} = \sqrt{\frac{T_{max}\,i/r - f_{r}mg}{\tfrac{1}{2}\rho C_{d}A}}$$

In [ ]:
# Every quantity comes from CAR, the parameter table in docs/HANDOVER.md. Nothing in this# notebook writes a plant parameter as a literal.c_drag = 0.5 * CAR.rho * CAR.CdA          # N per (m/s)^2F_roll = CAR.f_r * CAR.m * CAR.g          # N, flat roadF_max  = CAR.T_max * CAR.i / CAR.r        # N, full torque through the drivelinev_terminal = sqrt((F_max - F_roll) / c_drag)          # m/sprintln("drag coefficient      0.5*rho*CdA  = ", round(c_drag, digits = 4), " N/(m/s)^2")println("rolling force         f_r*m*g      = ", round(F_roll, digits = 2), " N")println("max tractive force    T_max*i/r    = ", round(F_max, digits = 2), " N")println("terminal speed                     = ", round(v_terminal, digits = 2), " m/s = ",        round(v_terminal * 3.6, digits = 1), " km/h")# The derived block of docs/HANDOVER.md quotes these; a mismatch means CAR and the handover# have drifted apart and every number downstream is suspect.@assert isapprox(c_drag, 0.378; atol = 1e-3)@assert isapprox(F_roll, 164.8; atol = 0.1)@assert isapprox(F_max, 1935.5; atol = 0.1)@assert isapprox(v_terminal, 68.4; atol = 0.1)

## 3. That number is too high — and saying so is the point**246 km/h.** No 1400 kg car with a 150 N.m engine does 246 km/h.The reason is in the name of the component: `IdealEngine` delivers its peak torque at everyspeed. A real engine delivers peak torque over a narrow band of crankshaft speed and much lesseverywhere else, and at 246 km/h in this gearing the crankshaft would be far past anywhere itmakes 150 N.m.This is worth stating out loud rather than leaving for a student to notice, for two reasons.The first is trust: someone who spots an obviously wrong number that nobody mentioned willreasonably distrust every number that comes after it. The second is that the missing piece isexactly the subject of lecture 2 — the engine torque map, $T_{max}(\omega)$ instead of$T_{max}$ — so the flaw is the hook into the next session rather than an embarrassment.Nothing in *this* lecture is harmed by it. Every scenario from here on runs between 90 and130 km/h, and *holding* any speed in that band takes 31 to 51 N.m — a third of the ceiling,where a real engine would have no trouble at all.

### The same balance, solved the other wayTerminal speed asks: given all the torque there is, how fast? The more useful question for acruise controller is the reverse — given a speed to hold, how much torque does it take? Set$\frac{dv}{dt} = 0$ again and solve for $T$ instead of $v$:$$T_{cruise}(v) \;=\; \left(\tfrac{1}{2}\rho C_{d}A\,v^{2} + f_{r}mg\right)\frac{r}{i}$$This is a *model inversion*: we have inverted the plant by hand to get the input that producesa wanted output. Notebook 02 runs exactly this number open loop and watches it work, and thenwatches it fail. Notebook 04 shows a PI controller arriving at the same number on its own,without ever being given a model.

In [ ]:
cruise_force(v)  = c_drag * v^2 + F_roll                   # N, v in m/scruise_torque(v) = cruise_force(v) * CAR.r / CAR.i         # N.mfor kmh in (90.0, 110.0, 130.0)    v = kmh / 3.6    println("hold ", rpad(kmh, 6), " km/h:  force ", lpad(round(cruise_force(v), digits = 1), 6),            " N   torque ", lpad(round(cruise_torque(v), digits = 2), 6), " N.m")end# The two numbers notebooks 02 and 04 come back to.@assert isapprox(cruise_torque(90 / 3.6), 31.1; atol = 0.05)@assert isapprox(cruise_torque(110 / 3.6), 40.1; atol = 0.05)

The step this lecture keeps returning to is 90 to 110 km/h, and *holding* the new speed costs40 N.m — a quarter of the engine's 150 N.m.Be careful with that number, because it is about to be misread. It is the torque needed to*hold* 110 km/h once the car is already there. It is not the torque a controller commands onthe way, which is set by the gain and the error, not by the force balance, and which is verymuch larger. A steady 40 N.m is a comfortable cruise; getting there in a few seconds instead ofa minute is not, and notebook 03 is where that bill arrives.

## 4. Open loop to terminal speedFull throttle from rest on flat road. No controller, no feedback, nothing watching the speed.The run is a Dyad analysis rather than a system assembled here, which matters for more thantidiness: the scenario — which sources drive which ports, what the car starts at, how long therun lasts — is part of the model, version-controlled next to it and tested with it. A notebookthat rebuilt the harness in Julia would be a second, untested copy of the same wiring.Expect this to be slow. Differentiating the drag term gives the restoring force per unit ofspeed error, $dF/dv = 2 \cdot 0.378\,v$, which at 100 km/h is about 21 N per m/s. Against1400 kg that is a time constant of roughly $m / (dF/dv) \approx 67$ s. **The open-loop car hasa time constant of about a minute.** Coasting from 110 down to 90 km/h really does take thatlong, and the time axes throughout this lecture have to be chosen to suit it.Keep that number in mind. When the loop is closed in notebook 03, the same car will settle ina few seconds. Feedback does not merely operate the system — it changes the system's dynamics,and a factor of twenty in speed of response is the most direct evidence of that we will see.

In [ ]:
# Scenarios are Dyad analyses. The harness, its sources, the initial conditions and the run# window are all in the model, so a scenario cell is a run and a plot and nothing else.Scenarios = VehicleSystemsComponents.Lecture1wot = Scenarios.WideOpenThrottleTransient()plt = plot_speed(wot; sig = "plant.v_kmh", label = "speed",                 title = "Full torque from rest, flat road")hline!(plt, [v_terminal * 3.6]; label = "terminal speed, analytic", ls = :dash, color = :grey)

The curve leaves the origin as a straight line and bends over into a horizontal asymptote. Bothhalves are the force balance being read out loud.At rest there is no drag, so the entire 1935 N goes into acceleration: $a = F/m = 1.38$ m/s².As speed builds, drag grows as $v^2$ and eats into the surplus, until at 68.4 m/s it hasconsumed all of it and the acceleration is zero. The car cannot go faster because there is noforce left to make it.If your curve does not flatten — if it runs off the top of the plot — the sign on the dragforce is wrong and it is adding energy instead of removing it.

## 5. The step testSlide 7's procedure, followed exactly:1. **Wait until the process is at rest.** The analysis starts the car already at 90 km/h with   the engine already delivering the torque that holds it, so it begins at rest in the sense   that matters — nothing is changing. Simulating the wait instead would cost six minutes of   simulated time to reach a state we can write down exactly.2. **Set the controller to manual.** There is no controller — the whole notebook is manual.3. **Change the control variable rapidly.** Step the torque from the 90 km/h cruise value to   the 110 km/h one, both computed above and both handed to the analysis as parameters.4. **Record the process variable and scale it by the change in the control variable.** That   division is what makes the gain `K` a property of the car rather than of the size of step we   happened to choose.The step is deliberately the one the rest of the lecture uses. Notebook 08 will tune acontroller from the three numbers this run produces and then apply it to this same step, so thetuning and the target match by construction.

In [ ]:
T_90, T_110 = cruise_torque(90 / 3.6), cruise_torque(110 / 3.6)t_step = 50.0# The analysis takes the two cruise torques as parameters, so the numbers derived on paper# above are the numbers the simulation actually runs. It also starts the car already settled# at 90 km/h rather than simulating the six minutes it would take to coast up to it.step_test = Scenarios.CarStepTestTransient(    tau_lo = T_90, tau_hi = T_110, v0 = 90 / 3.6, t_step = t_step)plot_speed(step_test; sig = "plant.v_kmh", label = "speed",           title = "Torque step $(round(T_90, digits = 1)) -> $(round(T_110, digits = 1)) N.m")

In [ ]:
# Commanded against delivered torque, zoomed onto the step itself. On the full 500 s axis# engine's lags are a single vertical line; they are the point of this plot, so the window is# a few seconds wide.plot_torque(step_test;    commanded = "cmd.y",    delivered = "plant.engine.limiter.y",    limit = nothing,    xlims = (t_step - 0.5, t_step + 3.0),    title = "The engine does not do as it is told, immediately")

Two distinct things are visible on that zoom, and they are different in kind.**Dead time.** For `theta_e` seconds after the command changes, the delivered torque does notmove *at all*. Not slowly — not at all. This is transport delay: injection to torque isroughly one engine cycle, and during that cycle nothing you ask for has any effect whatsoever.**Lag.** Then the torque starts to move, and approaches its new value exponentially with timeconstant `tau_e`. This is the manifold filling. The engine is responding, just not instantly.Dead time is the dangerous one. A lag makes a loop sluggish; dead time makes it *unstable*,because during the delay a feedback controller is acting on information about a past that itcan no longer change, and pushing harder makes it worse. Almost everything difficult in thislecture traces back to those `theta_e` seconds — and so does the possibility of notebook 08'sZiegler-Nichols test, which needs a plant that can be driven to oscillate.

## 6. The three responsesSlide 6 splits any response into two pieces, and names their sum:$$\underbrace{y(t)}_{\text{dynamic response}}\;=\;\underbrace{y_{transient}(t)}_{\text{dies out}}\;+\;\underbrace{y_{steady}(t)}_{\text{what is left}}$$On the speed plot above:- **Steady state** — the flat stretch before the step, and the flat stretch at the end. The  transient has died out and the output has settled into a fixed relationship with the input.  This is the region that steady-state error is measured in, which is what notebook 03 is about.- **Transient** — the S-shaped climb between them. Temporary by definition: it is the system's  reaction to an abrupt change, and it decays. This is the region that overshoot, rise time and  settling time are measured in, which is what notebooks 04 and 05 are about.- **Dynamic response** — the whole trace. The entire journey from one settled state to the next.Almost every disagreement about whether a controller is any good is a disagreement about whichof the first two matters more. A loop that is fast and overshoots has a good steady state andan ugly transient. A loop that creeps to the right answer has the opposite. There is no singleright answer, which is why tuning is a craft and not a formula — even though notebook 08proceeds to give you three formulas.

## 7. Reading K, tau and theta off the curveThree numbers describe a first-order-plus-dead-time (FOPTD) plant, and they are exactly whatnotebook 08's tuning tables consume:$$G(s) \;=\; \frac{K\,e^{-\theta s}}{\tau s + 1}$$- **`K`, the gain** — how far the output moves per unit of input, in the steady state.  km/h per N.m here. It is the change in speed divided by the change in torque, which is step 4  of slide 7's procedure.- **`tau`, the time constant** — how long the output takes to get there. Dominated by the  vehicle's own mass-against-drag time constant, the 67 s from section 4, not by the engine's  0.3 s.- **`theta`, the dead time** — how long nothing happens first. This is the engine's transport  delay `theta_e`.`fopdt_fit` reads them off the trace by the construction in `tuning_methods.pdf`: find thetimes at which the output has covered half and $1 - 1/e$ of its total change, and solve for thestraight-line intercept those two points imply. The step size and step time are handed to itrather than guessed, because we built the step and know them exactly.

In [ ]:
K, tau, theta = fopdt_fit(step_test; output = "plant.v_kmh", A = T_110 - T_90, t0 = t_step)println("measured off the step response")println("  K     = ", round(K, digits = 3), " (km/h) per N.m")println("  tau   = ", round(tau, digits = 1), " s")println("  theta = ", round(theta, digits = 3), " s")println()# The local (tangent) gain and time constant, differentiating the steady-state force balance:# dT/dv = 2*c_drag*v*r/i, and tau_local = m/(2*c_drag*v). Both depend on speed, which is the# whole point of printing them at each end of the step.local_gain(v) = 3.6 / (2 * c_drag * v * CAR.r / CAR.i)     # (km/h) per N.mlocal_tau(v)  = CAR.m / (2 * c_drag * v)                   # sprintln("computed from the force balance, at each end of the step")for kmh in (90.0, 110.0)    v = kmh / 3.6    println("  at ", rpad(kmh, 6), " km/h:  gain ", round(local_gain(v), digits = 3),            " (km/h) per N.m   time constant ", round(local_tau(v), digits = 1), " s")endprintln("  secant gain across the whole step  ", round(20 / (T_110 - T_90), digits = 3))println("  engine transport delay             ", CAR.theta_e, " s")

Two of those three land where the force balance says they should, and the third does not.**`K` and `tau` are bracketed by the analytic numbers.** The measured gain sits between thetangent gain at 90 km/h and the one at 110 km/h, close to the secant across the step — which isexactly what a gain read between two settled states *is*. The measured time constant likewisesits between the two local ones. Both are speed-dependent because drag is quadratic, so neitherhas a single true value; the fit returns an average over the interval we tested.**`theta` does not.** It comes out around 1.6 s — roughly forty times the engine's `theta_e` of0.04 s. The engine's transport delay is real and it is in there, but it is not what dominatesthis measurement.What dominates is that **the car is not a linear system and the FOPTD form assumes it is.**Between 90 and 110 km/h the local time constant falls from about 74 s to about 61 s, so theresponse is still accelerating relative to a true exponential when it passes the landmarks thefit reads. The construction has only one parameter that can absorb "slower at first than anexponential", and that parameter is the dead time.The cleanest evidence is what happens when the step is made smaller. Shrink it to 1 km/h andthe fitted `theta` falls to about 0.6 s; shrink it to 0.2 km/h and it goes *negative*. Anegative transport delay is physically impossible, which settles the question: at that scalethe construction is measuring the curvature of the response, not a delay.So the three numbers are a description of this car over this step, not a set of physicalconstants. That is not a defect in the method and it is not a reason to distrust the result —it is the reason Cohen-Coon and Ziegler-Nichols are called *heuristics*. Notebook 08 feedsthese three numbers into tuning tables that were fitted to real industrial processes withexactly this kind of slop in them, and the gains that come out are a starting point to beadjusted, never a final answer.

## What this bought usWe now have a car we understand from first principles, three numbers that describe it, and oneuncomfortable fact: **left alone, it does whatever the road tells it to.** The open-loop runheld its speed only because nothing disturbed it.That is the opening of **notebook 02**. We take the cruise torque computed above — the modelinversion, 31 N.m for 90 km/h — apply it open loop, and watch it hold the speed perfectly onflat dry road. Then we put a hill under it, and a headwind, and 200 kg of luggage, and watchit settle at the wrong speed every time, with nothing anywhere in the architecture able tonotice that anything is wrong.Which is the argument for feedback, and the reason the other eight notebooks exist.